# PDF to Markdown Converter (VS Code + Colab)

Optimized for VS Code Colab Extension workflow.

## Quick Start
1. **Upload PDF**: Right-click your PDF in VS Code → "Upload to Colab server"
2. **Run this notebook**: Execute cells in order
3. **Get output**: Markdown appears in `/content/output/` (visible in VS Code workspace)

## Tools Comparison

| Tool | Best For | Tables | Math | Speed |
|------|----------|--------|------|-------|
| **Marker** | Clean PDFs, books | Excellent | Good | Fastest |
| **Docling** | Mixed content | Great | Good | Medium |
| **Nougat** | Academic, equations | Basic | Excellent | Slow |

In [ ]:
#@title 1. Environment Setup { display-mode: "form" }
#@markdown Checks GPU and detects uploaded PDFs

import torch
import os
from pathlib import Path

# GPU Check
print("=" * 50)
print("ENVIRONMENT CHECK")
print("=" * 50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_memory:.1f} GB)")
else:
    print("WARNING: No GPU detected!")
    print("Enable GPU: Runtime > Change runtime type > GPU")

# Create output directory
OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Auto-detect PDFs in /content/
print("\n" + "=" * 50)
print("PDF DETECTION")
print("=" * 50)

pdf_files = list(Path("/content").glob("*.pdf"))

if pdf_files:
    print(f"Found {len(pdf_files)} PDF(s):\n")
    for i, pdf in enumerate(pdf_files):
        size_mb = pdf.stat().st_size / 1e6
        print(f"  [{i+1}] {pdf.name} ({size_mb:.1f} MB)")
    
    # Store for later use
    DETECTED_PDFS = pdf_files
else:
    print("No PDFs found in /content/")
    print("\nUpload via VS Code:")
    print("  Right-click PDF > 'Upload to Colab server'")
    DETECTED_PDFS = []

print("\n" + "=" * 50)

In [ ]:
#@title 2. Select PDF & Tool { display-mode: "form" }
#@markdown Interactive selection with recommendations

from pathlib import Path

def select_pdf(pdf_list):
    """Select PDF from detected files."""
    if not pdf_list:
        print("ERROR: No PDFs detected. Upload first, then re-run cell 1.")
        return None
    
    if len(pdf_list) == 1:
        print(f"Auto-selected: {pdf_list[0].name}")
        return str(pdf_list[0])
    
    print("Multiple PDFs found. Select one:")
    for i, pdf in enumerate(pdf_list):
        print(f"  [{i+1}] {pdf.name}")
    
    choice = input("\nEnter number: ").strip()
    try:
        idx = int(choice) - 1
        if 0 <= idx < len(pdf_list):
            return str(pdf_list[idx])
    except (ValueError, IndexError):
        pass
    
    print("Invalid selection. Using first PDF.")
    return str(pdf_list[0])


def get_tool_recommendation():
    """Interactive questionnaire for tool selection."""
    
    print("\n" + "=" * 50)
    print("TOOL RECOMMENDATION QUIZ")
    print("=" * 50)
    
    questions = [
        {
            "q": "Primary content type?",
            "opts": [
                ("1", "Heavy math/equations", {"nougat": 3, "marker": 1}),
                ("2", "Many tables/data", {"docling": 3, "marker": 2}),
                ("3", "Mostly text", {"marker": 3, "docling": 2}),
                ("4", "Mixed content", {"docling": 3, "marker": 2, "nougat": 1})
            ]
        },
        {
            "q": "PDF type?",
            "opts": [
                ("1", "Scanned/image-based", {"nougat": 2, "docling": 2}),
                ("2", "Digital (selectable text)", {"marker": 2, "docling": 1}),
                ("3", "Mixed/unsure", {"docling": 2})
            ]
        },
        {
            "q": "Priority?",
            "opts": [
                ("1", "Speed", {"marker": 3, "docling": 1}),
                ("2", "Quality", {"nougat": 2, "docling": 2}),
                ("3", "Balance", {"docling": 2, "marker": 2})
            ]
        }
    ]
    
    scores = {"docling": 0, "marker": 0, "nougat": 0}
    
    for i, q in enumerate(questions):
        print(f"\n{i+1}. {q['q']}")
        for key, label, _ in q["opts"]:
            print(f"   [{key}] {label}")
        
        choice = input("   Choice: ").strip()
        
        for key, _, score_dict in q["opts"]:
            if choice == key:
                for tool, pts in score_dict.items():
                    scores[tool] += pts
                break
    
    # Determine winner
    recommended = max(scores, key=scores.get)
    
    tool_meta = {
        "docling": ("Docling", "Mixed content, good tables", "~2-3 min/100pg"),
        "marker": ("Marker", "Fast, clean layouts", "~1-2 min/100pg"),
        "nougat": ("Nougat", "Academic, math-heavy", "~5-10 min/100pg")
    }
    
    print("\n" + "=" * 50)
    print("RECOMMENDATION")
    print("=" * 50)
    
    name, desc, time_est = tool_meta[recommended]
    print(f"\n  Tool: {name.upper()}")
    print(f"  Why: {desc}")
    print(f"  Est. Time: {time_est}")
    
    print("\n  Scores:")
    for tool in sorted(scores, key=lambda x: -scores[x]):
        bar = "*" * scores[tool]
        print(f"    {tool_meta[tool][0]:10} {bar}")
    
    # Allow override
    print("\n" + "-" * 50)
    print("Override? [1] Marker  [2] Docling  [3] Nougat  [Enter] Accept")
    override = input("Choice: ").strip()
    
    if override == "1":
        return "marker"
    elif override == "2":
        return "docling"
    elif override == "3":
        return "nougat"
    
    return recommended


# Run selection
PDF_PATH = select_pdf(DETECTED_PDFS)
if PDF_PATH:
    SELECTED_TOOL = get_tool_recommendation()
    print(f"\n{'='*50}")
    print("READY TO CONVERT")
    print("=" * 50)
    print(f"  PDF: {Path(PDF_PATH).name}")
    print(f"  Tool: {SELECTED_TOOL.upper()}")
    print(f"\n  Run the next cell to start conversion.")

In [ ]:
#@title 3. Install & Convert { display-mode: "form" }
#@markdown Installs selected tool and runs conversion

import subprocess
import time as time_module
from pathlib import Path

def convert_with_marker(pdf_path, output_dir):
    """Convert using Marker."""
    print("Installing Marker...")
    subprocess.run(["pip", "install", "-q", "marker-pdf"], check=True)
    
    print("\nConverting with Marker...")
    result = subprocess.run(
        ["marker_single", pdf_path, "--output_dir", output_dir],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
    else:
        print(result.stdout)
    
    # Find output
    md_files = list(Path(output_dir).rglob("*.md"))
    return md_files[0] if md_files else None


def convert_with_docling(pdf_path, output_dir):
    """Convert using Docling."""
    print("Installing Docling...")
    subprocess.run(["pip", "install", "-q", "docling"], check=True)
    
    print("\nConverting with Docling...")
    from docling.document_converter import DocumentConverter
    
    converter = DocumentConverter()
    result = converter.convert(pdf_path)
    markdown = result.document.export_to_markdown()
    
    # Save output
    output_name = Path(pdf_path).stem + "_docling.md"
    output_path = Path(output_dir) / output_name
    output_path.write_text(markdown, encoding="utf-8")
    
    return output_path


def convert_with_nougat(pdf_path, output_dir):
    """Convert using Nougat."""
    print("Installing Nougat...")
    subprocess.run(["pip", "install", "-q", "nougat-ocr"], check=True)
    
    print("\nConverting with Nougat (this takes longer)...")
    print("Note: First run downloads model (~1.5GB)\n")
    result = subprocess.run(
        ["nougat", pdf_path, "-o", output_dir, "--no-skipping"],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
    else:
        print(result.stdout)
    
    # Find output
    mmd_files = list(Path(output_dir).rglob("*.mmd"))
    if mmd_files:
        # Rename .mmd to .md
        md_path = mmd_files[0].with_suffix(".md")
        mmd_files[0].rename(md_path)
        return md_path
    return None


# Verify selections exist
if 'PDF_PATH' not in dir() or not PDF_PATH:
    print("ERROR: Run cell 2 first to select PDF.")
elif 'SELECTED_TOOL' not in dir() or not SELECTED_TOOL:
    print("ERROR: Run cell 2 first to select tool.")
else:
    print("=" * 50)
    print(f"CONVERTING: {Path(PDF_PATH).name}")
    print(f"USING: {SELECTED_TOOL.upper()}")
    print("=" * 50 + "\n")
    
    start_time = time_module.time()
    
    # Run appropriate converter
    converters = {
        "marker": convert_with_marker,
        "docling": convert_with_docling,
        "nougat": convert_with_nougat
    }
    
    OUTPUT_FILE = converters[SELECTED_TOOL](PDF_PATH, OUTPUT_DIR)
    
    elapsed = time_module.time() - start_time
    
    print("\n" + "=" * 50)
    print("CONVERSION COMPLETE")
    print("=" * 50)
    print(f"  Time: {elapsed/60:.1f} minutes")
    
    if OUTPUT_FILE and Path(OUTPUT_FILE).exists():
        size_kb = Path(OUTPUT_FILE).stat().st_size / 1024
        print(f"  Output: {OUTPUT_FILE}")
        print(f"  Size: {size_kb:.1f} KB")
        print(f"\n  File is now visible in VS Code workspace!")
        print(f"  Location: /content/output/")
    else:
        print("  WARNING: Output file not found. Check logs above.")

In [ ]:
#@title 4. Preview Output { display-mode: "form" }
#@markdown Shows first 3000 characters of converted markdown

from pathlib import Path

# Fallback if cell 1 wasn't run
if 'OUTPUT_DIR' not in dir():
    OUTPUT_DIR = "/content/output"

def preview_markdown(output_dir, chars=3000):
    """Preview the converted markdown."""
    md_files = list(Path(output_dir).rglob("*.md"))
    
    if not md_files:
        print("No markdown files found. Run conversion first.")
        return
    
    # Get most recent
    latest = max(md_files, key=lambda p: p.stat().st_mtime)
    content = latest.read_text(encoding="utf-8")
    
    print(f"FILE: {latest.name}")
    print(f"TOTAL: {len(content):,} characters")
    print("=" * 50)
    print(content[:chars])
    
    if len(content) > chars:
        print(f"\n... [truncated, showing {chars}/{len(content):,} chars]")

preview_markdown(OUTPUT_DIR)

In [ ]:
#@title 5. Quality Check { display-mode: "form" }
#@markdown Analyzes conversion quality and provides stats

from pathlib import Path
import re

# Fallback if cell 1 wasn't run
if 'OUTPUT_DIR' not in dir():
    OUTPUT_DIR = "/content/output"

def analyze_quality(output_dir):
    """Analyze markdown quality."""
    md_files = list(Path(output_dir).rglob("*.md"))
    
    if not md_files:
        print("No markdown files found.")
        return
    
    latest = max(md_files, key=lambda p: p.stat().st_mtime)
    content = latest.read_text(encoding="utf-8")
    
    # Analysis
    stats = {
        "Characters": len(content),
        "Words": len(content.split()),
        "Lines": content.count("\n") + 1,
        "Headings": len(re.findall(r"^#{1,6}\s", content, re.MULTILINE)),
        "Tables": content.count("|---"),
        "Code blocks": content.count("```") // 2,
        "Images": len(re.findall(r"!\[.*?\]\(.*?\)", content)),
        "Math blocks": content.count("$$") // 2 + content.count("\\["),
        "Links": len(re.findall(r"\[.*?\]\(.*?\)", content)) - len(re.findall(r"!\[.*?\]\(.*?\)", content)),
    }
    
    # Potential issues
    issues = []
    
    # Check for garbled text (high ratio of special chars)
    special_ratio = len(re.findall(r"[^\w\s]", content)) / max(len(content), 1)
    if special_ratio > 0.15:
        issues.append("High special character ratio (possible OCR issues)")
    
    # Check for very short content
    if stats["Words"] < 1000:
        issues.append("Very short output (conversion may have failed)")
    
    # Check for broken tables
    table_starts = content.count("|")
    if stats["Tables"] > 0 and table_starts / stats["Tables"] < 10:
        issues.append("Possible broken table formatting")
    
    print("=" * 50)
    print("QUALITY ANALYSIS")
    print("=" * 50)
    print(f"\nFile: {latest.name}\n")
    
    print("Content Statistics:")
    for key, val in stats.items():
        print(f"  {key:15} {val:,}")
    
    print("\nQuality Indicators:")
    if not issues:
        print("  No issues detected")
    else:
        for issue in issues:
            print(f"  WARNING: {issue}")
    
    # Estimated pages (rough)
    est_pages = stats["Words"] / 300  # ~300 words per page
    print(f"\nEstimated original pages: ~{int(est_pages)}")

analyze_quality(OUTPUT_DIR)

In [ ]:
#@title 6. Download Options { display-mode: "form" }
#@markdown Download via browser or view path for VS Code access

from pathlib import Path

# Fallback if cell 1 wasn't run
if 'OUTPUT_DIR' not in dir():
    OUTPUT_DIR = "/content/output"

md_files = list(Path(OUTPUT_DIR).rglob("*.md"))

if md_files:
    latest = max(md_files, key=lambda p: p.stat().st_mtime)
    
    print("=" * 50)
    print("OUTPUT ACCESS")
    print("=" * 50)
    
    print("\n[Option 1] VS Code Workspace Mount")
    print("  Your file is already accessible in VS Code!")
    print(f"  Path: {latest}")
    print("  Tip: Click refresh icon in VS Code Colab workspace")
    
    print("\n[Option 2] Browser Download")
    print("  Run the cell below to trigger download.")
    
    print("\n" + "=" * 50)
else:
    print("No markdown files found.")

In [ ]:
#@title 6b. Trigger Browser Download { display-mode: "form" }
#@markdown Downloads the markdown file directly to your computer

from google.colab import files
from pathlib import Path

# Fallback if cell 1 wasn't run
if 'OUTPUT_DIR' not in dir():
    OUTPUT_DIR = "/content/output"

md_files = list(Path(OUTPUT_DIR).rglob("*.md"))

if md_files:
    latest = max(md_files, key=lambda p: p.stat().st_mtime)
    print(f"Downloading: {latest.name}")
    files.download(str(latest))
else:
    print("No markdown files to download.")

---
## Troubleshooting

| Issue | Solution |
|-------|----------|
| No PDF detected | Right-click PDF in VS Code → "Upload to Colab server" |
| Out of memory | Use Marker (most efficient) or restart runtime |
| Poor quality | Try different tool (Nougat for math, Docling for tables) |
| Slow conversion | Nougat is 5-10x slower than Marker by design |
| File not in VS Code | Click refresh icon in Colab workspace panel |

---
## Next Steps

After conversion, your markdown is ready for:
1. **Chunking** with LlamaIndex
2. **Embedding** and storage in Helix-DB
3. **Querying** via MCP + Claude Code